In [1]:
import jax
import jax.numpy as jnp
from custom_jax import forces
import numpy as np

Running cmake --build & --install in /home/jens/repos/custom-jax/build


In [2]:
@jax.jit
def potential_via_matrix(x, eps=1e-2):
    rij2 = jnp.sum((x[:, None, :] - x[None, :, :]) ** 2, axis=-1)
    rinv = jnp.where(rij2 > 0, 1. / jnp.sqrt(rij2 + eps**2), 0.)
    
    return -jnp.sum(rinv, axis=1)

ffipot = jax.jit(forces.potential, static_argnames=("block_size", "eps"))

In [3]:
x = jnp.arange(3*1024).reshape(-1,3) * 1.

phi0 = potential_via_matrix(x, eps=1e-2)
phi1 = ffipot(x, 1, eps=1e-2)
phi2 = ffipot(x, 16, eps=1e-2)

print(phi0[0:5])
print(phi1[0:5])
print(np.allclose(phi1, phi0, rtol=1e-4))
print(np.allclose(phi1, phi2))
# print(phi)

[-1.4449532 -1.6372147 -1.7332516 -1.7972128 -1.8451369]
[-1.4449844 -1.6372452 -1.7332764 -1.7972336 -1.8451614]
True
True


In [4]:
x = jax.random.normal(jax.random.PRNGKey(0), (128*1024, 3)).block_until_ready()
%timeit -n 10 potential_via_matrix(x).block_until_ready()
%timeit -n 10 ffipot(x, block_size=16).block_until_ready()
%timeit -n 10 ffipot(x, block_size=32).block_until_ready()
%timeit -n 10 ffipot(x, block_size=64).block_until_ready()
%timeit -n 10 ffipot(x, block_size=128).block_until_ready()
%timeit -n 10 ffipot(x, block_size=256).block_until_ready()

140 ms ± 4.84 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
67.1 ms ± 1.28 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
37.1 ms ± 1.13 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
36.2 ms ± 1.02 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
37 ms ± 1.04 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
38.1 ms ± 1.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
print((128*1024)**2 / 0.036  / 1e9)

477.21858844444444


In [5]:
x = jax.random.normal(jax.random.PRNGKey(0), (256*1024, 3)).block_until_ready()
%timeit -n 1 potential_via_matrix(x).block_until_ready()
%timeit -n 1 ffipot(x, block_size=16).block_until_ready()
%timeit -n 1 ffipot(x, block_size=32).block_until_ready()
%timeit -n 1 ffipot(x, block_size=64).block_until_ready()
%timeit -n 1 ffipot(x, block_size=128).block_until_ready()
%timeit -n 1 ffipot(x, block_size=256).block_until_ready()

632 ms ± 142 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
268 ms ± 12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
147 ms ± 14.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
143 ms ± 11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
143 ms ± 12.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
142 ms ± 10.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
print((256*1024)**2 / 0.143 / 1e9)

480.5557813706294
